<a href="https://colab.research.google.com/github/V1LL4pro/Deep-Learning/blob/Corte-1/T3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Descarga Dataset

In [ ]:
import numpy as np
import os
from PIL import Image
from tqdm import tqdm
import requests
import urllib.parse

# Configuración inicial
BASE_PATH = "/content/quickdraw_dataset"
MAX_IMAGES_PER_CLASS = 1000

# Obtener la lista completa de categorías
categories_url = "https://raw.githubusercontent.com/googlecreativelab/quickdraw-dataset/master/categories.txt"
response = requests.get(categories_url)
original_categories = [line.strip() for line in response.text.split('\n') if line.strip()]

# Crear estructura de directorios con nombres válidos
os.makedirs(BASE_PATH, exist_ok=True)
for category in original_categories:
    safe_name = category.replace(" ", "_")
    os.makedirs(os.path.join(BASE_PATH, safe_name), exist_ok=True)

def process_category(original_name):
    try:
        # Generar nombres seguros para URL y sistema de archivos
        url_name = urllib.parse.quote(original_name)
        dir_name = original_name.replace(" ", "_")

        # Descargar el archivo .npy
        url = f"https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{url_name}.npy"
        response = requests.get(url, stream=True)
        response.raise_for_status()

        # Guardar temporalmente
        temp_path = os.path.join(BASE_PATH, f"{dir_name}.npy")
        with open(temp_path, "wb") as f:
            f.write(response.content)

        # Procesar imágenes
        images = np.load(temp_path)
        np.random.shuffle(images)

        # Guardar imágenes en formato PNG
        for i, img_data in enumerate(images[:MAX_IMAGES_PER_CLASS]):
            img = Image.fromarray(img_data.reshape(28, 28).astype(np.uint8))
            img.save(os.path.join(BASE_PATH, dir_name, f"{dir_name}_{i}.png"))

        # Limpiar archivo temporal
        os.remove(temp_path)

    except Exception as e:
        print(f"Error en {original_name}: {str(e)}")

# Procesar todas las categorías
for category in tqdm(original_categories):
    process_category(category)

print("Proceso completado exitosamente!")

 70%|██████▉   | 240/345 [18:25<07:26,  4.26s/it]

In [ ]:
!pip install keras-tuner

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from keras_tuner import RandomSearch
import matplotlib.pyplot as plt
import os

# 1. Carga y preparación de datos
def load_dataset(base_path, num_classes=345, max_per_class=MAX_IMAGES_PER_CLASS):
    images = []
    labels = []
    class_names = sorted(os.listdir(base_path))

    for label_idx, class_name in enumerate(class_names[:num_classes]):
        class_path = os.path.join(base_path, class_name)
        print(f"Cargando {class_name}...")

        for i, img_file in enumerate(os.listdir(class_path)[:max_per_class]):
            img_path = os.path.join(class_path, img_file)
            img = keras.preprocessing.image.load_img(
                img_path, color_mode='grayscale', target_size=(28, 28))
            img_array = keras.preprocessing.image.img_to_array(img)
            images.append(img_array)
            labels.append(label_idx)

    return np.array(images), np.array(labels)

# Cargar datos
X, y = load_dataset('/content/quickdraw_dataset', num_classes=345, max_per_class=1000)

# Preprocesamiento
X = X / 255.0  # Normalización
y = keras.utils.to_categorical(y)  # One-hot encoding

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# 2. Construcción del modelo base
def build_model(hp=None):
    model = keras.Sequential()
    model.add(layers.Flatten(input_shape=(28, 28, 1)))

    # Hiperparámetros a ajustar
    if hp:
        num_layers = hp.Int('num_layers', 2, 4)
        units = hp.Int('units', min_value=128, max_value=512, step=128)
        learning_rate = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
    else:
        num_layers = 3
        units = 256
        learning_rate = 1e-3

    for _ in range(num_layers):
        model.add(layers.Dense(units, activation='relu'))

    model.add(layers.Dense(345, activation='softmax'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# 3. Búsqueda de hiperparámetros
tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='tuner_dir',
    project_name='quickdraw_mlp'
)

tuner.search(
    X_train, y_train,
    epochs=20,
    validation_data=(X_val, y_val),
    batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(patience=3)]
)

# 4. Entrenamiento del mejor modelo
best_model = tuner.get_best_models(num_models=1)[0]
history = best_model.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val),
    batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(patience=5)]
)

# 5. Evaluación final
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print(f"\nExactitud en prueba: {test_acc:.4f}")

# 6. Visualización de resultados
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Exactitud durante el entrenamiento')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida durante el entrenamiento')
plt.legend()
plt.show()

# Guardar modelo
best_model.save('best_mlp_model.h5')